# 通过时间反向传播 （backpropagation through time，BPTT）
:label:`sec_bptt`

到目前为止，我们已经反复提到像*梯度爆炸*或*梯度消失*，
以及需要对循环神经网络*分离梯度*。
例如，在 :numref:`sec_rnn_scratch`中，
我们在序列上调用了`detach`函数。
为了能够快速构建模型并了解其工作原理，
上面所说的这些概念都没有得到充分的解释。
本节将更深入地探讨序列模型反向传播的细节，
以及相关的数学原理。

当我们首次实现循环神经网络（ :numref:`sec_rnn_scratch`）时，
遇到了梯度爆炸的问题。
如果做了练习题，就会发现梯度截断对于确保模型收敛至关重要。
为了更好地理解此问题，本节将回顾序列模型梯度的计算方式，
它的工作原理没有什么新概念，毕竟我们使用的仍然是链式法则来计算梯度。

我们在 :numref:`sec_backprop`中描述了多层感知机中的
前向与反向传播及相关的计算图。
循环神经网络中的前向传播相对简单。
*通过时间反向传播*（backpropagation through time，BPTT）
 :cite:`Werbos.1990`实际上是循环神经网络中*反向传播技术的一个特定应用*。
**它要求我们将循环神经网络的计算图一次展开一个时间步，以获得模型变量和参数之间的依赖关系**。
然后，基于链式法则，应用反向传播来计算和存储梯度。
由于序列可能相当长，因此依赖关系也可能相当长。
例如，某个1000个字符的序列，
其第一个词元可能会对最后位置的词元产生重大影响。
这在计算上是不可行的（它需要的时间和内存都太多了），
并且还需要超过1000个矩阵的乘积才能得到非常难以捉摸的梯度。
这个过程充满了计算与统计的不确定性。
在下文中，我们将阐明会发生什么以及如何在实践中解决它们。

## 循环神经网络的梯度分析
:label:`subsec_bptt_analysis`

我们从一个描述循环神经网络工作原理的简化模型开始，
此模型忽略了隐状态的特性及其更新方式的细节。
这里的数学表示没有像过去那样明确地区分标量、向量和矩阵，
因为这些细节对于分析并不重要，
反而只会使本小节中的符号变得混乱。

在这个简化模型中，我们将时间步$t$的隐状态表示为$h_t$，
输入表示为$x_t$，输出表示为$o_t$。
回想一下我们在 :numref:`subsec_rnn_w_hidden_states`中的讨论，
输入$x_t$和隐状态$h_{t-1}$可以拼接后与隐藏层中的一个权重变量相乘。
因此，我们分别使用$w_h$和$w_o$来表示隐藏层和输出层的权重。
每个时间步的隐状态和输出可以写为：

$$\begin{aligned}h_t &= f(x_t, h_{t-1}, w_h),\\o_t &= g(h_t, w_o),\end{aligned}$$
:eqlabel:`eq_bptt_ht_ot`

其中$f$和$g$分别是隐藏层和输出层的变换。
因此，我们有一个链
$\{\ldots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_t), \ldots\}$，
它们通过循环计算彼此依赖。
**前向传播**相当简单，一次一个时间步的遍历三元组$(x_t, h_t, o_t)$，
然后通过一个目标函数在所有$T$个时间步内
评估输出$o_t$和对应的标签$y_t$之间的差异：

$$L(x_1, \ldots, x_T, y_1, \ldots, y_T, w_h, w_o) = \frac{1}{T}\sum_{t=1}^T l(y_t, o_t).$$

对于反向传播，问题则有点棘手，
特别是当我们计算目标函数$L$关于参数$w_h$的梯度时。
具体来说，按照链式法则：

$$\begin{aligned}\frac{\partial L}{\partial w_h}  & = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_h}  \\& = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_o)}{\partial h_t}  \frac{\partial h_t}{\partial w_h}.\end{aligned}$$
:eqlabel:`eq_bptt_partial_L_wh`

在 :eqref:`eq_bptt_partial_L_wh`中乘积的第一项和第二项很容易计算，
而第三项$\partial h_t/\partial w_h$是使事情变得棘手的地方，
因为我们需要***循环地计算参数$w_h$对$h_t$的影响***。
根据 :eqref:`eq_bptt_ht_ot`中的递归计算，
$h_t$既依赖于$h_{t-1}$又依赖于$w_h$，
其中$h_{t-1}$的计算也依赖于$w_h$。
因此，使用链式法则产生：

$$\frac{\partial h_t}{\partial w_h}= \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial w_h} +\frac{\partial f(x_{t},h_{t-1},w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}.$$
:eqlabel:`eq_bptt_partial_ht_wh_recur`

为了导出上述梯度，假设我们有三个序列$\{a_{t}\},\{b_{t}\},\{c_{t}\}$，
当$t=1,2,\ldots$时，序列满足$a_{0}=0$且$a_{t}=b_{t}+c_{t}a_{t-1}$。
对于$t\geq 1$，就很容易得出：

$$a_{t}=b_{t}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t}c_{j}\right)b_{i}.$$
:eqlabel:`eq_bptt_at`

基于下列公式替换$a_t$、$b_t$和$c_t$：

$$\begin{aligned}a_t &= \frac{\partial h_t}{\partial w_h},\\
b_t &= \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial w_h}, \\
c_t &= \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial h_{t-1}},\end{aligned}$$

公式 :eqref:`eq_bptt_partial_ht_wh_recur`中的梯度计算
满足$a_{t}=b_{t}+c_{t}a_{t-1}$。
因此，对于每个 :eqref:`eq_bptt_at`，
我们可以使用下面的公式移除 :eqref:`eq_bptt_partial_ht_wh_recur`中的循环计算

$$\frac{\partial h_t}{\partial w_h}=\frac{\partial f(x_{t},h_{t-1},w_h)}{\partial w_h}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t} \frac{\partial f(x_{j},h_{j-1},w_h)}{\partial h_{j-1}} \right) \frac{\partial f(x_{i},h_{i-1},w_h)}{\partial w_h}.$$
:eqlabel:`eq_bptt_partial_ht_wh_gen`

虽然我们可以使用链式法则递归地计算$\partial h_t/\partial w_h$，
但当$t$很大时这个链就会变得很长。
我们需要想想办法来处理这一问题.



## ✅ 问题 1：  
> **“$w_h$ 不是应该在时间步 $t$ 内更新 $t$ 次么？为什么这里完全没有提到？”**

### 🔍 回答：
**不，$w_h$ 在一个序列的前向/反向传播过程中是固定的，不会在时间步内更新多次。**

### 📌 关键区分：**计算梯度 vs. 更新参数**
- **前向传播**（Forward）：固定 $w_h$，计算所有 $h_1, h_2, ..., h_T$
- **反向传播**（BPTT）：固定 $w_h$，**计算整个序列对 $w_h$ 的总梯度 $\frac{\partial L}{\partial w_h}$**
- **参数更新**：在**整个序列处理完后**，用优化器（如 SGD）**一次性更新 $w_h$**：
  $$
  w_h \leftarrow w_h - \eta \cdot \frac{\partial L}{\partial w_h}
  $$

> 💡 **RNN 的训练单位是一个完整序列**（或一个 batch of sequences），不是单个时间步。

### ❌ 常见误解：
- 误以为每个时间步都独立更新 $w_h$ → 这会导致训练不稳定、破坏序列依赖
- 实际上，**梯度是跨时间步累积的**，更新是**序列级**（或 batch 级）的

✅ 所以原文**不提更新**，因为它只关注**如何计算梯度**（BPTT 的核心），而非优化器行为。

---

## ✅ 问题 2：  
> **“$h_{t-1}$ 和 $w_h$ 不是独立的吗？为什么会出现 $\frac{\partial f}{\partial h_{t-1}} \cdot \frac{\partial h_{t-1}}{\partial w_h}$？”**

### 🔍 回答：
**$h_{t-1}$ 和 $w_h$ 并不独立！$h_{t-1}$ 本身就是 $w_h$ 的函数。**

### 🧠 递归依赖关系：
$$
\begin{aligned}
h_1 &= f(x_1, h_0, w_h) \\
h_2 &= f(x_2, h_1, w_h) = f(x_2, f(x_1, h_0, w_h), w_h) \\
h_3 &= f(x_3, h_2, w_h) = f(x_3, f(x_2, f(x_1, h_0, w_h), w_h), w_h) \\
&\vdots
\end{aligned}
$$

> **每个 $h_t$ 都显式或隐式地依赖于 $w_h$**！

### 📐 链式法则的必然结果：
当我们对 $h_t = f(x_t, h_{t-1}, w_h)$ 求 $\frac{\partial h_t}{\partial w_h}$ 时：
- **直接依赖**：$f$ 显式包含 $w_h$ → $\frac{\partial f}{\partial w_h}$
- **间接依赖**：$f$ 通过 $h_{t-1}$ 依赖 $w_h$ → $\frac{\partial f}{\partial h_{t-1}} \cdot \frac{\partial h_{t-1}}{\partial w_h}$

这正是多元函数链式法则的标准形式：
$$
\frac{\partial h_t}{\partial w_h} = 
\underbrace{\frac{\partial f}{\partial w_h}}_{\text{direct}} +
\underbrace{\frac{\partial f}{\partial h_{t-1}}}_{\text{how } h_t \text{ changes with } h_{t-1}} \cdot 
\underbrace{\frac{\partial h_{t-1}}{\partial w_h}}_{\text{how } h_{t-1} \text{ depends on } w_h}
$$

✅ **所以 $h_{t-1}$ 和 $w_h$ 不独立——这是 RNN 长期依赖和梯度消失/爆炸的根源！**

---

## ✅ 问题 3：  
> **“如果使用递推公式 $a_t = b_t + c_t a_{t-1}$，其中 $b_{t-1}, ..., b_0$ 等项去哪里了？”**

### 🔍 回答：
**它们没有消失！而是被展开到求和项中了。** 公式 (eq_bptt_at) 正是这个展开的结果。

### 🧮 递推展开示例：
给定：
- $a_0 = 0$
- $a_1 = b_1 + c_1 a_0 = b_1$
- $a_2 = b_2 + c_2 a_1 = b_2 + c_2 b_1$
- $a_3 = b_3 + c_3 a_2 = b_3 + c_3 b_2 + c_3 c_2 b_1$
- $a_4 = b_4 + c_4 b_3 + c_4 c_3 b_2 + c_4 c_3 c_2 b_1$

>一般形式：
>$$
>a_t = b_t + \sum_{i=1}^{t-1} \left( \prod_{j=i+1}^t c_j \right) b_i
>$$

### 📌 对应到梯度：
- $a_t = \frac{\partial h_t}{\partial w_h}$
- $b_i = \frac{\partial f(x_i, h_{i-1}, w_h)}{\partial w_h}$（时间步 $i$ 对 $w_h$ 的**直接贡献**）
- $\prod_{j=i+1}^t c_j = \prod_{j=i+1}^t \frac{\partial f(x_j, h_{j-1}, w_h)}{\partial h_{j-1}}$（从 $i+1$ 到 $t$ 的**路径导数乘积**）

> ✅ 所有历史 $b_i$（即每个时间步对 $w_h$ 的直接偏导）都保留在求和中！


### 🎯 物理意义解释：
- **第一项**：当前时间步 $t$ 对 $w_h$ 的**直接敏感度**
- **求和项**：所有过去时间步 $i < t$ 对 $w_h$ 的**间接影响**
  - $\frac{\partial f(x_i, ...)}{\partial w_h}$：时间步 $i$ 的直接贡献
  - $\prod_{j=i+1}^t \frac{\partial f}{\partial h_{j-1}}$：这个影响如何通过 $h_{i} \to h_{i+1} \to \cdots \to h_t$ 传递过来

> ⚠️ **梯度消失/爆炸的根源就在这里**：  
> 如果 $\left| \frac{\partial f}{\partial h} \right| < 1$，乘积 $\prod c_j \to 0$（梯度消失）  
> 如果 $\left| \frac{\partial f}{\partial h} \right| > 1$，乘积 $\prod c_j \to \infty$（梯度爆炸）


### 完全计算 ###

显然，我们可以仅仅计算 :eqref:`eq_bptt_partial_ht_wh_gen`中的全部总和，
然而，这样的计算非常缓慢，并且可能会发生梯度爆炸，
因为初始条件的微小变化就可能会对结果产生巨大的影响。
也就是说，我们可以观察到类似于蝴蝶效应的现象，
即**初始条件的很小变化就会导致结果发生不成比例的变化**。
这对于我们想要估计的模型而言是非常不可取的。
毕竟，我们正在寻找的是能够很好地泛化高稳定性模型的估计器。
因此，在实践中，这种方法几乎从未使用过。

### 截断时间步 ###

或者，我们可以在$\tau$步后截断
 :eqref:`eq_bptt_partial_ht_wh_gen`中的求和计算。
这是我们到目前为止一直在讨论的内容，
例如在 :numref:`sec_rnn_scratch`中分离梯度时。
这会带来真实梯度的*近似*，
只需将求和终止为$\partial h_{t-\tau}/\partial w_h$。
在实践中，这种方式工作得很好。
它通常被称为截断的通过时间反向传播 :cite:`Jaeger.2002`。
这样做导致**该模型主要侧重于短期影响，而不是长期影响**。
这在现实中是可取的，因为它会将估计值偏向更简单和更稳定的模型。

### 随机截断 ###

最后，我们可以用一个随机变量替换$\partial h_t/\partial w_h$，
该随机变量在预期中是正确的，但是会截断序列。
这个随机变量是通过使用序列$\xi_t$来实现的，
序列预定义了$0 \leq \pi_t \leq 1$，
其中$P(\xi_t = 0) = 1-\pi_t$且$P(\xi_t = \pi_t^{-1}) = \pi_t$，
因此$E[\xi_t] = 1$。
我们使用它来替换 :eqref:`eq_bptt_partial_ht_wh_recur`中的
梯度$\partial h_t/\partial w_h$得到：

$$z_t= \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial w_h} +\xi_t \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}.$$

从$\xi_t$的定义中推导出来$E[z_t] = \partial h_t/\partial w_h$。
每当$\xi_t = 0$时，递归计算终止在这个$t$时间步。
这导致了不同长度序列的加权和，其中长序列出现的很少，
所以将适当地加大权重。
这个想法是由塔莱克和奥利维尔 :cite:`Tallec.Ollivier.2017`提出的。



### ✅ 问题 1：  
> **为什么 $\xi_t$ 要服从这样的分布：**  
> $$
P(\xi_t = 0) = 1 - \pi_t,\quad P(\xi_t = \pi_t^{-1}) = \pi_t
$$

### 🔍 核心目的：**构造一个无偏**（unbiased）

我们知道，完整的梯度递推是：
$$
\frac{\partial h_t}{\partial w_h} = b_t + c_t \cdot \frac{\partial h_{t-1}}{\partial w_h}
\quad \text{其中 } b_t = \frac{\partial f}{\partial w_h},\ c_t = \frac{\partial f}{\partial h_{t-1}}
$$

但当 $t$ 很大时，这个递归链很长，计算开销大，且容易梯度消失/爆炸。

于是，我们想**以某种概率提前终止递归**（即设 $\frac{\partial h_{t-1}}{\partial w_h} = 0$），但又**不能引入偏差**（bias）。

### 🎯 如何做到“无偏截断”？

我们引入随机变量 $\xi_t$，用它乘以递归项：
$$
z_t = b_t + \xi_t \cdot c_t \cdot \frac{\partial h_{t-1}}{\partial w_h}
$$

我们希望：
$$
\mathbb{E}[z_t] = \frac{\partial h_t}{\partial w_h} = b_t + c_t \cdot \frac{\partial h_{t-1}}{\partial w_h}
$$

即：
$$
\mathbb{E}[\xi_t] \cdot c_t \cdot \frac{\partial h_{t-1}}{\partial w_h} = c_t \cdot \frac{\partial h_{t-1}}{\partial w_h}
\quad \Rightarrow \quad \mathbb{E}[\xi_t] = 1
$$

所以，**只要 $\mathbb{E}[\xi_t] = 1$，$z_t$ 就是无偏估计**！

### 💡 为什么选这个特定分布？

我们有两个自由度：取值和概率。设计目标：
1. **有时截断**（$\xi_t = 0$）→ 节省计算
2. **有时放大**（$\xi_t > 1$）→ 补偿截断带来的期望损失
3. **保证 $\mathbb{E}[\xi_t] = 1$**

最简单的二值分布满足：
- 以概率 $1 - \pi_t$ 设 $\xi_t = 0$（**截断**）
- 以概率 $\pi_t$ 设 $\xi_t = a$（**放大**）

则期望：
$$
\mathbb{E}[\xi_t] = (1 - \pi_t) \cdot 0 + \pi_t \cdot a = \pi_t a
$$
令其等于 1：
$$
\pi_t a = 1 \quad \Rightarrow \quad a = \frac{1}{\pi_t}
$$

✅ 所以：
$$
P(\xi_t = 0) = 1 - \pi_t,\quad P\left(\xi_t = \frac{1}{\pi_t}\right) = \pi_t
\quad \Rightarrow \quad \mathbb{E}[\xi_t] = 1
$$

> 📌 这就是该分布的**数学必然性**：它是满足“以概率 $1-\pi_t$ 截断，同时保持无偏”的最简单二值分布。

---

### ✅ 问题 2：  
> **$\xi_t$ 在递推公式中充当什么角色？有什么功能？**

### 🧩 角色：**随机开关 + 重要性权重**

#### 功能 1：**随机截断长依赖链**
- 当 $\xi_t = 0$ 时：
  $$
  z_t = b_t + 0 = b_t
  $$
  → **忽略历史梯度 $\frac{\partial h_{t-1}}{\partial w_h}$**
  → **递归在此终止**！不再需要计算更早的梯度
- 这大大**减少计算量**，尤其对长序列

#### 功能 2：**无偏补偿**（Importance Weighting）
- 当 $\xi_t = \frac{1}{\pi_t}$ 时（发生概率为 $\pi_t$）：
  $$
  z_t = b_t + \frac{1}{\pi_t} \cdot c_t \cdot \frac{\partial h_{t-1}}{\partial w_h}
  $$
  → 对保留的路径进行**放大**，使得**期望仍等于真实梯度**

> 💡 这类似于 **蒙特卡洛积分中的重要性采样**（Importance Sampling）：
> - 稀有事件（长路径）被赋予高权重
> - 常见事件（短路径）被频繁采样但权重低
> - 整体期望不变

#### 功能 3：**隐式正则化 & 缓解梯度爆炸**
- 随机截断相当于对 RNN 施加了一种**随机深度**（stochastic depth）或**dropout-like** 正则化
- 减少对极长依赖的过度拟合
- 实践中可提升泛化能力

## 📊 直观理解：加权平均不同长度的路径

完整梯度包含所有历史路径：
$$
\frac{\partial h_t}{\partial w_h} = b_t + c_t b_{t-1} + c_t c_{t-1} b_{t-2} + \cdots
$$

使用 $\xi$ 后，每次只保留**一条随机截断的路径**，例如：
- 以高概率在 $t-2$ 截断 → 只算 $b_t + c_t b_{t-1}$
- 以低概率保留到 $t-5$ → 算 $b_t + c_t b_{t-1} + \cdots + c_t \cdots c_{t-4} b_{t-5}$，但该项被放大 $1/\pi$ 倍

多次采样后，**期望等于完整梯度**，但**单次计算成本大幅降低**。


## ⚠️ 补充说明：$\pi_t$ 如何选择？

- $\pi_t$ 通常随 $t$ **指数衰减**，例如 $\pi_t = \gamma^t$（$\gamma < 1$）
- 这反映了一个先验：**越远的历史，越不重要**
- 也可以设为常数（如 $\pi_t = 0.5$），但会牺牲更多长程信息

---

## ✅ 总结

| 问题 | 答案 |
|------|------|
| **1. 为什么 $\xi_t$ 服从该分布**？ | 为了满足 $\mathbb{E}[\xi_t] = 1$，从而保证 $z_t$ 是 $\frac{\partial h_t}{\partial w_h}$ 的**无偏估计**，同时允许以概率 $1-\pi_t$ **截断递归**以节省计算 |
| **2. $\xi_t$ 的角色和功能**？ | - **随机开关**：以概率 $1-\pi_t$ 终止梯度回传- **重要性权重**：以概率 $\pi_t$ 保留路径并放大 $1/\pi_t$ 倍，保持期望无偏- **隐式正则化**：减少对长程依赖的过拟合 |

> 🎯 **本质**：这是一种**计算效率与统计无偏之间的巧妙权衡**，属于**随机优化**（stochastic optimization）在序列模型中的精妙应用。

### 比较策略

![比较RNN中计算梯度的策略，3行自上而下分别为：随机截断、常规截断、完整计算](../img/truncated-bptt.svg)
:label:`fig_truncated_bptt`

 :numref:`fig_truncated_bptt`说明了
当基于循环神经网络使用通过时间反向传播
分析《时间机器》书中前几个字符的三种策略：

* 第一行采用随机截断，方法是将文本划分为不同长度的片断；
* 第二行采用常规截断，方法是将文本分解为相同长度的子序列。
  这也是我们在循环神经网络实验中一直在做的；
* 第三行采用通过时间的完全反向传播，结果是产生了在计算上不可行的表达式。

遗憾的是，虽然随机截断在理论上具有吸引力，
但很可能是由于多种因素在实践中并不比常规截断更好。
- 首先，在对过去若干个时间步经过反向传播后，
观测结果足以捕获实际的依赖关系。
- 其次，增加的方差抵消了时间步数越多梯度越精确的事实。
- 第三，我们真正想要的是只有短范围交互的模型。

因此，模型需要的正是截断的通过时间反向传播方法所具备的**轻度正则化**效果。



## 通过时间反向传播的细节

在讨论一般性原则之后，我们看一下通过时间反向传播问题的细节。
与 :numref:`subsec_bptt_analysis`中的分析不同，
下面我们将展示如何计算**目标函数相对于所有分解模型参数的梯度**。
为了保持简单，我们考虑一个没有偏置参数的循环神经网络，
其在隐藏层中的激活函数使用恒等映射（$\phi(x)=x$）。
对于时间步$t$，设单个样本的输入及其对应的标签分别为
$\mathbf{x}_t \in \mathbb{R}^d$和$y_t$。
计算隐状态$\mathbf{h}_t \in \mathbb{R}^h$和
输出$\mathbf{o}_t \in \mathbb{R}^q$的方式为：

$$\begin{aligned}\mathbf{h}_t &= \mathbf{W}_{hx} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1},\\
\mathbf{o}_t &= \mathbf{W}_{qh} \mathbf{h}_{t},\end{aligned}$$

其中权重参数为$\mathbf{W}_{hx} \in \mathbb{R}^{h \times d}$、
$\mathbf{W}_{hh} \in \mathbb{R}^{h \times h}$和
$\mathbf{W}_{qh} \in \mathbb{R}^{q \times h}$。
用$l(\mathbf{o}_t, y_t)$表示时间步$t$处
（即从序列开始起的超过$T$个时间步）的损失函数，
则我们的目标函数的总体损失是：

$$L = \frac{1}{T} \sum_{t=1}^T l(\mathbf{o}_t, y_t).$$

为了在循环神经网络的计算过程中可视化模型变量和参数之间的依赖关系，
我们可以为模型绘制一个计算图，
如 :numref:`fig_rnn_bptt`所示。
例如，时间步3的隐状态$\mathbf{h}_3$的计算
取决于模型参数$\mathbf{W}_{hx}$和$\mathbf{W}_{hh}$，
以及最终时间步的隐状态$\mathbf{h}_2$
以及当前时间步的输入$\mathbf{x}_3$。

![上图表示具有三个时间步的循环神经网络模型依赖关系的计算图。未着色的方框表示变量，着色的方框表示参数，圆表示运算符](../img/rnn-bptt.svg)
:label:`fig_rnn_bptt`

正如刚才所说， :numref:`fig_rnn_bptt`中的模型参数是
$\mathbf{W}_{hx}$、$\mathbf{W}_{hh}$和$\mathbf{W}_{qh}$。
通常，训练该模型需要对这些参数进行梯度计算：
$\partial L/\partial \mathbf{W}_{hx}$、
$\partial L/\partial \mathbf{W}_{hh}$和
$\partial L/\partial \mathbf{W}_{qh}$。
根据 :numref:`fig_rnn_bptt`中的依赖关系，
我们可以沿箭头的相反方向遍历计算图，依次计算和存储梯度。
为了灵活地表示链式法则中不同形状的矩阵、向量和标量的乘法，
我们继续使用如 :numref:`sec_backprop`中
所述的$\text{prod}$运算符。

首先，在任意时间步$t$，
目标函数关于模型输出的微分计算是相当简单的：

$$\frac{\partial L}{\partial \mathbf{o}_t} =  \frac{\partial l (\mathbf{o}_t, y_t)}{T \cdot \partial \mathbf{o}_t} \in \mathbb{R}^q.$$
:eqlabel:`eq_bptt_partial_L_ot`

现在，我们可以计算目标函数关于输出层中参数$\mathbf{W}_{qh}$的梯度：
$\partial L/\partial \mathbf{W}_{qh} \in \mathbb{R}^{q \times h}$。
基于 :numref:`fig_rnn_bptt`，
目标函数$L$**通过且仅通过**$\mathbf{o}_1, \ldots, \mathbf{o}_T$
依赖于$\mathbf{W}_{qh}$。
依据链式法则，得到

$$
\frac{\partial L}{\partial \mathbf{W}_{qh}}
= \sum_{t=1}^T \text{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{W}_{qh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{o}_t} \mathbf{h}_t^\top,
$$

其中$\partial L/\partial \mathbf{o}_t$是
由 :eqref:`eq_bptt_partial_L_ot`给出的。

接下来，如 :numref:`fig_rnn_bptt`所示，
在最后的时间步$T$，目标函数$L$仅通过$\mathbf{o}_T$
依赖于隐状态$\mathbf{h}_T$。
因此，我们通过使用链式法可以很容易地得到梯度
$\partial L/\partial \mathbf{h}_T \in \mathbb{R}^h$：

$$\frac{\partial L}{\partial \mathbf{h}_T} = \text{prod}\left(\frac{\partial L}{\partial \mathbf{o}_T}, \frac{\partial \mathbf{o}_T}{\partial \mathbf{h}_T} \right) = \mathbf{W}_{qh}^\top \frac{\partial L}{\partial \mathbf{o}_T}.$$
:eqlabel:`eq_bptt_partial_L_hT_final_step`

当目标函数$L$通过$\mathbf{h}_{t+1}$和$\mathbf{o}_t$
依赖$\mathbf{h}_t$时，
对任意时间步$t < T$来说都变得更加棘手。
根据链式法则，隐状态的梯度
$\partial L/\partial \mathbf{h}_t \in \mathbb{R}^h$
在任何时间步骤$t < T$时都可以递归地计算为：

$$\frac{\partial L}{\partial \mathbf{h}_t} = \text{prod}\left(\frac{\partial L}{\partial \mathbf{h}_{t+1}}, \frac{\partial \mathbf{h}_{t+1}}{\partial \mathbf{h}_t} \right) + \text{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{h}_t} \right) = \mathbf{W}_{hh}^\top \frac{\partial L}{\partial \mathbf{h}_{t+1}} + \mathbf{W}_{qh}^\top \frac{\partial L}{\partial \mathbf{o}_t}.$$
:eqlabel:`eq_bptt_partial_L_ht_recur`

为了进行分析，对于任何时间步$1 \leq t \leq T$展开递归计算得

$$\frac{\partial L}{\partial \mathbf{h}_t}= \sum_{i=t}^T {\left(\mathbf{W}_{hh}^\top\right)}^{T-i} \mathbf{W}_{qh}^\top \frac{\partial L}{\partial \mathbf{o}_{T+t-i}}.$$
:eqlabel:`eq_bptt_partial_L_ht`

我们可以从 :eqref:`eq_bptt_partial_L_ht`中看到，
这个简单的线性例子已经展现了长序列模型的一些关键问题：
它陷入到$\mathbf{W}_{hh}^\top$的潜在的非常大的幂。
在这个幂中，小于1的特征值将会消失，大于1的特征值将会发散。
这在数值上是不稳定的，表现形式为梯度消失或梯度爆炸。
解决此问题的一种方法是按照计算方便的需要截断时间步长的尺寸
如 :numref:`subsec_bptt_analysis`中所述。
实际上，这种截断是通过在给定数量的时间步之后分离梯度来实现的。
稍后，我们将学习更复杂的序列模型（如长短期记忆模型）
是如何进一步缓解这一问题的。

最后， :numref:`fig_rnn_bptt`表明：
目标函数$L$通过隐状态$\mathbf{h}_1, \ldots, \mathbf{h}_T$
依赖于隐藏层中的模型参数$\mathbf{W}_{hx}$和$\mathbf{W}_{hh}$。
为了计算有关这些参数的梯度
$\partial L / \partial \mathbf{W}_{hx} \in \mathbb{R}^{h \times d}$和$\partial L / \partial \mathbf{W}_{hh} \in \mathbb{R}^{h \times h}$，
我们应用链式规则得：

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}_{hx}}
&= \sum_{t=1}^T \text{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_{hx}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{x}_t^\top,\\
\frac{\partial L}{\partial \mathbf{W}_{hh}}
&= \sum_{t=1}^T \text{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_{hh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{h}_{t-1}^\top,
\end{aligned}
$$

其中$\partial L/\partial \mathbf{h}_t$
是由 :eqref:`eq_bptt_partial_L_hT_final_step`和
 :eqref:`eq_bptt_partial_L_ht_recur`递归计算得到的，
是影响数值稳定性的关键量。

正如我们在 :numref:`sec_backprop`中所解释的那样，
由于通过时间反向传播是反向传播在循环神经网络中的应用方式，
所以训练循环神经网络交替使用前向传播和通过时间反向传播。
通过时间反向传播依次计算并存储上述梯度。
具体而言，存储的中间值会被重复使用，以避免重复计算，
例如存储$\partial L/\partial \mathbf{h}_t$，
以便在计算$\partial L / \partial \mathbf{W}_{hx}$和
$\partial L / \partial \mathbf{W}_{hh}$时使用。

## 小结

* “通过时间反向传播”仅仅适用于反向传播在具有隐状态的序列模型。
* 截断是计算方便性和数值稳定性的需要。截断包括：规则截断和随机截断。
* 矩阵的高次幂可能导致神经网络特征值的发散或消失，将以梯度爆炸或梯度消失的形式表现。
* 为了计算的效率，“通过时间反向传播”在计算期间会缓存中间值。



## 练习

1. 假设我们拥有一个对称矩阵$\mathbf{M} \in \mathbb{R}^{n \times n}$，其特征值为$\lambda_i$，对应的特征向量是$\mathbf{v}_i$（$i = 1, \ldots, n$）。通常情况下，假设特征值的序列顺序为$|\lambda_i| \geq |\lambda_{i+1}|$。
   1. 证明$\mathbf{M}^k$拥有特征值$\lambda_i^k$。
   1. 证明对于一个随机向量$\mathbf{x} \in \mathbb{R}^n$，$\mathbf{M}^k \mathbf{x}$将有较高概率与$\mathbf{M}$的特征向量$\mathbf{v}_1$在一条直线上。形式化这个证明过程。
   1. 上述结果对于循环神经网络中的梯度意味着什么？
1. 除了梯度截断，还有其他方法来应对循环神经网络中的梯度爆炸吗？

[Discussions](https://discuss.d2l.ai/t/2107)
